# Single-Agent Berth Rescheduling — Baseline

**1. Data preparation → 2. LLM agent (detect + resolve) → 3. Independent validation**

One LLM agent receives the port calls of each conflict-affected berth, finds the conflicts itself and proposes a shifted timetable.
Python never tells the agent *which* vessels conflict or *how* to move them; it only validates the result afterwards.

| # | Success criterion | How it is checked |
|---|---|---|
| 1 | Detect conflicts correctly | agent pairs vs. reference pairs (precision / recall / F1) |
| 2 | Resolve conflicts | no original conflict remains |
| 3 | Minimize added delay | every move is *tight* (cannot start earlier) + compared with a greedy FCFS reference |
| 4 | Valid rescheduled timetable | all batches answered, valid JSON, known & unique IDs, parseable dates |
| 5 | ETA before ETD | every row of the final timetable |
| 6 | Berth & modules preserved | final vs. original |
| 7 | Service duration preserved | final vs. original |
| 8 | No new conflicts | conflicts in final ∖ original conflicts = ∅ |
| 9 | ETA not earlier than original | every row of the final timetable |
| 10 | Final validation | criteria 2 and 4–9 all pass |

## 1. Data Preparation

In [1]:
%pip -q install openai pandas

In [2]:
import os, re, io, csv, json, time
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from IPython.display import display

FILE = 'escales_arribades_2025 3.csv'
if not os.path.exists(FILE):                      # Colab: ask for the CSV
    from google.colab import files
    print('Upload the Port of Barcelona CSV:')
    FILE = next(iter(files.upload()))

# تم إضافة on_bad_lines='skip' و engine='python' لتجاوز أسطر الملف التالفة
raw = pd.read_csv(FILE, dtype='string', encoding='utf-8-sig', on_bad_lines='skip', engine='python')
raw.columns = raw.columns.str.strip().str.upper()
REQUIRED = ['ESCALANUM', 'MOLLCODI', 'MOLLMODULS', 'ETA', 'ETD']
missing = [c for c in REQUIRED if c not in raw.columns]
assert not missing, f'Missing columns: {missing}'


@lru_cache(maxsize=None)
def modules_as_set(value):
    """'3-6' -> {3,4,5,6};  '1,4' -> {1,4}"""
    s = str(value).replace(' ', '')
    if not re.fullmatch(r'\d+(-\d+)?([,;]\d+(-\d+)?)*', s):
        raise ValueError(f'Invalid modules: {value}')
    result = set()
    for piece in re.split('[,;]', s):
        a, _, b = piece.partition('-')
        a, b = int(a), int(b or a)
        if a > b or b - a > 200:
            raise ValueError(f'Invalid module range: {value}')
        result.update(range(a, b + 1))
    return frozenset(result)


def valid_modules(value):
    try:
        return bool(modules_as_set(value))
    except ValueError:
        return False


def find_conflicts(table):
    """All pairs on the same berth that share a module and overlap in time (touching is OK)."""
    pairs = set()
    for _, group in table.groupby('berth'):
        active = []
        for row in group.sort_values('eta').itertuples(index=False):
            active = [old for old in active if old.etd > row.eta]
            for old in active:
                if modules_as_set(row.modules) & modules_as_set(old.modules):
                    pairs.add(tuple(sorted((str(old.vessel_id), str(row.vessel_id)))))
            active.append(row)
    return pairs


# ---- cleaning ------------------------------------------------------------
df = raw[REQUIRED].copy()
df.columns = ['vessel_id', 'berth', 'modules', 'eta', 'etd']
for col in ['vessel_id', 'berth', 'modules']:
    df[col] = df[col].str.strip()
for col in ['eta', 'etd']:
    df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed').dt.floor('min')
df = df.dropna()
df = df[(df.vessel_id != '') & (df.berth != '') & (df.etd > df.eta)]
df = df[df.modules.map(valid_modules)].drop_duplicates()
df = df[~df.vessel_id.duplicated(keep=False)].sort_values(['berth', 'eta', 'vessel_id']).reset_index(drop=True)

# ---- sanity tests of the conflict rule -------------------------------------
def _t(rows):
    t = pd.DataFrame(rows, columns=['vessel_id', 'berth', 'modules', 'eta', 'etd'])
    return t.assign(eta=pd.to_datetime(t.eta), etd=pd.to_datetime(t.etd))
assert find_conflicts(_t([['A', 'B1', '1-3', '2025-01-01 08:00', '2025-01-01 12:00'],
                          ['B', 'B1', '3-5', '2025-01-01 10:00', '2025-01-01 14:00']])) == {('A', 'B')}
assert find_conflicts(_t([['A', 'B1', '1', '2025-01-01 08:00', '2025-01-01 12:00'],
                          ['B', 'B1', '1', '2025-01-01 12:00', '2025-01-01 16:00']])) == set()  # touching
assert find_conflicts(_t([['A', 'B1', '1', '2025-01-01 08:00', '2025-01-01 12:00'],
                          ['B', 'B2', '1', '2025-01-01 10:00', '2025-01-01 14:00']])) == set()  # other berth

print(f'Original rows: {len(raw):,} | Cleaned rows: {len(df):,} | Berths: {df.berth.nunique()}')
display(df.head())

Original rows: 10,309 | Cleaned rows: 8,675 | Berths: 36


,vessel_id,berth,modules,eta,etd
0,71718-1,01C,20-23,2025-02-01 19:49:00,2025-02-03 22:03:00
1,71725-1,01C,13-16,2025-02-03 19:42:00,2025-02-03 22:43:00
2,71719-1,01C,13-16,2025-02-04 18:49:00,2025-02-04 22:07:00
3,71919-1,01C,20-23,2025-02-05 19:15:00,2025-02-05 22:24:00
4,71915-1,01C,20-23,2025-02-06 19:20:00,2025-02-06 21:59:00


### Batching
One big request overflows the model output, and long batches make the agent hallucinate conflicts.
So each conflict-affected berth is cut into **small batches of 25 vessels** (sorted by ETA):

* only berths that contain at least one overlap are sent (berth-level filter — the agent is **not** told which vessels conflict);
* batches of the same berth are handled one after another (berths run in parallel); vessels from earlier batches that are
  **still at the quay** when a batch starts are added as `fixed=1` context rows **with the times the agent already gave them**,
  so no conflict is split between batches and a delay cannot spill unseen into the next batch;
* modules are written out explicitly (`3-6` → `3,4,5,6`) so the agent can compare them directly.

In [3]:
BATCH_ROWS = 25


def batch_to_csv(batch):
    buf = io.StringIO()
    w = csv.writer(buf, lineterminator='\n')
    w.writerow(['vessel_id', 'berth', 'modules', 'eta', 'etd', 'duration_min', 'fixed'])
    for r in batch.itertuples(index=False):
        w.writerow([r.vessel_id, r.berth, ','.join(map(str, sorted(modules_as_set(r.modules)))),
                    r.eta.strftime('%Y-%m-%d %H:%M'), r.etd.strftime('%Y-%m-%d %H:%M'),
                    int((r.etd - r.eta).total_seconds() // 60), r.fixed])
    return buf.getvalue()


affected_berths = sorted({b for b, g in df.groupby('berth') if find_conflicts(g)})
agent_input = df[df.berth.isin(affected_berths)]
n_batches = sum(-(-n // BATCH_ROWS) for n in agent_input.groupby('berth').size())

print(f'Berths with overlaps: {len(affected_berths)} | Rows sent to the agent: {len(agent_input):,} '
      f'| Batches: {n_batches}')
display(agent_input.groupby('berth').size().rename('vessels').to_frame().T)

Berths with overlaps: 15 | Rows sent to the agent: 5,817 | Batches: 240


berth,01C,01D,19A,20A,20D,22B,22C,24B,29A,32C,32D,32G,34B,36A,90A
vessels,956,802,454,359,550,39,91,485,273,104,110,137,122,1316,19


## 2. Single LLM Agent

In [5]:
from openai import OpenAI

#MODEL    = 'deepseek-reasoner'
MODEL    = 'deepseek-chat'
BASE_URL = 'https://api.deepseek.com'
PRICE_IN, PRICE_OUT = 0.28, 0.42
MAX_ATTEMPTS, MAX_WORKERS = 3, 16

try:
    from google.colab import userdata
    API_KEY = userdata.get('DEEPSEEK_API_KEY')
except Exception:
    API_KEY = os.environ.get('DEEPSEEK_API_KEY')
assert API_KEY, 'Add DEEPSEEK_API_KEY to Colab Secrets (or the environment).'

client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=300, max_retries=0)

PROMPT = """You are a vessel berth scheduling agent.
You receive port calls of ONE berth as CSV: vessel_id, berth, modules, eta, etd, duration_min, fixed.
`modules` lists every quay module the vessel occupies. Rows with fixed=1 are context only (vessels from the previous batch, shown at their
final times): never move them and do not list conflicts between two fixed rows.

CONFLICT = two vessels that share at least one module number AND whose stays overlap in time:
eta_A < etd_B and eta_B < etd_A. Touching stays (etd_A == eta_B) are NOT conflicts.
Many vessels are at the berth at the same time on DIFFERENT modules - that is normal and NOT a conflict.
For every candidate pair, write down the shared module numbers; if there are none, it is not a conflict.

TASK 1 - DETECT: list every conflicting pair in the ORIGINAL schedule.
TASK 2 - RESOLVE: delay vessels (fixed=0 only) so that the final schedule has NO conflicts.
Every conflict you list must be resolved by your changes.

Rules for every vessel you move:
- Keep its berth and modules.
- Only move forward: new_eta >= original eta.
- Keep its duration exactly: new_etd = new_eta + duration_min.
- Do not create a conflict with any other vessel (moved, unmoved or fixed).
- Minimize total added delay: move as few vessels as possible, move the vessel that needs the smaller
  delay, and give it the EARLIEST feasible new_eta (normally the etd of the vessel that blocks it).
- After every move re-check the moved vessel against all later vessels on shared modules.

Reply with ONLY this JSON:
{"conflicts": [{"a": "vessel_id_A", "b": "vessel_id_B", "shared_modules": [3, 4]}],
 "changes": [{"vessel_id": "ID", "new_eta": "YYYY-MM-DD HH:MM", "new_etd": "YYYY-MM-DD HH:MM"}],
 "notes": "one short sentence"}
List only moved vessels in "changes". Use empty lists when there are no conflicts."""


def parse_json(text):
    """Accept a bare JSON object, also when wrapped in ```json fences."""
    text = (text or '').strip()
    answer = json.loads(text[text.find('{'):text.rfind('}') + 1])
    if not isinstance(answer, dict):
        raise ValueError('reply is not a JSON object')
    return answer


def run_agent(batch):
    """One agent call for one batch (retried if the reply is not valid JSON)."""
    log = {'berth': batch.berth.iloc[0], 'rows': len(batch), 'attempts': 0,
           'tokens_in': 0, 'tokens_out': 0, 'answer': None, 'error': None}
    for _ in range(MAX_ATTEMPTS):
        log['attempts'] += 1
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=0, max_tokens=32768 if 'reasoner' in MODEL else 8192,
                **({} if 'reasoner' in MODEL else {'response_format': {'type': 'json_object'}}),
                messages=[{'role': 'system', 'content': PROMPT},
                          {'role': 'user', 'content': batch_to_csv(batch)}])
            if r.usage:
                log['tokens_in'] += r.usage.prompt_tokens
                log['tokens_out'] += r.usage.completion_tokens
            log['answer'], log['error'] = parse_json(r.choices[0].message.content), None
            break
        except Exception as exc:
            log['error'] = f'{type(exc).__name__}: {exc}'
    return log


def run_berth(group):
    """The agent works through one berth batch by batch; later batches see its earlier decisions."""
    group = group.sort_values(['eta', 'vessel_id']).copy()
    logs = []
    for k in range(0, len(group), BATCH_ROWS):
        own, before = group.iloc[k:k + BATCH_ROWS], group.iloc[:k]
        carry = before[before.etd > own.eta.iloc[0]]
        log = run_agent(pd.concat([carry.assign(fixed=1), own.assign(fixed=0)]))
        for ch in (log['answer'] or {}).get('changes') or []:
            try:
                i = own.index[own.vessel_id == ch['vessel_id']][0]
                group.loc[i, ['eta', 'etd']] = pd.Timestamp(ch['new_eta']), pd.Timestamp(ch['new_etd'])
            except Exception:
                pass
        logs.append(log)
    return logs


start = time.perf_counter()
with ThreadPoolExecutor(MAX_WORKERS) as pool:
    logs = [log for berth_logs in pool.map(run_berth, [g for _, g in agent_input.groupby('berth')])
            for log in berth_logs]
seconds = round(time.perf_counter() - start, 1)

failed = [l for l in logs if l['answer'] is None]
answer = {'conflicts_identified': [p for l in logs if l['answer'] for p in (l['answer'].get('conflicts') or [])],
          'rescheduling_changes': [c for l in logs if l['answer'] for c in (l['answer'].get('changes') or [])]}

print(f'{MODEL} | {len(logs)} batches in {seconds}s | failed batches: {len(failed)}')
print(f"Reported conflict pairs: {len(answer['conflicts_identified'])} | "
      f"Proposed changes: {len(answer['rescheduling_changes'])}")
for l in failed:
    print('  FAILED', l['berth'], '-', l['error'][:150])
display(pd.DataFrame(answer['rescheduling_changes']).head(10))

deepseek-chat | 240 batches in 295.0s | failed batches: 0
Reported conflict pairs: 2646 | Proposed changes: 1976


,vessel_id,new_eta,new_etd,new_eta_placeholder
0,73862-1,2025-04-23 23:11,2025-04-24 01:25,NaN
1,73866-1,2025-04-25 22:22,2025-04-26 00:30,NaN
2,73836-1,2025-04-30 15:37,2025-05-02 18:42,NaN
3,73042-1,2025-05-02 18:42,2025-05-03 11:08,NaN
4,73867-1,2025-05-03 11:08,2025-05-03 13:27,NaN
5,73042-3,2025-05-03 13:27,2025-05-03 18:20,NaN
6,73868-1,2025-05-03 18:20,2025-05-03 20:37,NaN
7,74074-1,2025-05-03 20:37,2025-05-04 00:19,NaN
8,73847-1,2025-05-04 00:19,2025-05-04 04:02,NaN
9,73869-1,2025-05-04 04:02,2025-05-04 06:25,NaN


In [6]:
import time
import json
from openai import OpenAI


agent_replies = []


for berth_id, group_df in df.groupby('berth'):

    dataset_text = group_df.to_csv(index=False)

    attempts = 0
    max_attempts = 3
    success = False
    answer = None
    api_error = None

    while attempts < max_attempts and not success:
        attempts += 1
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': PROMPT},
                    {'role': 'user', 'content': 'Here is the conflict-affected schedule subset CSV:\n' + dataset_text}
                ],
                response_format={'type': 'json_object'},
                temperature=0,
                max_tokens=8192
            )

            content_output = response.choices[0].message.content
            if not content_output:
                raise ValueError('Model returned empty response.')

            answer = json.loads(content_output)
            success = True

        except Exception as exc:
            api_error = str(exc)

            if "429" in api_error or "RateLimit" in api_error:
                time.sleep(3 * attempts)
            else:
                time.sleep(1)


    agent_replies.append({
        "berth": berth_id,
        "rows": len(group_df),
        "attempts": attempts,
        "answer": answer,
        "error": api_error
    })


    time.sleep(1.5)

print("تم الانتهاء من معالجة جميع الأرصفة بنجاح مع تفعيل نظام التأخير والتحكم بالضغط!")

تم الانتهاء من معالجة جميع الأرصفة بنجاح مع تفعيل نظام التأخير والتحكم بالضغط!


## 3. Independent Validation
Everything below is computed **after** the agent has finished.
The agent's changes are applied exactly as proposed (never repaired) and the resulting timetable is checked against all ten criteria.

In [8]:
original_pairs = find_conflicts(df)
violations, echo_changed = [], []

# ---- criterion 1: detection ---------------------------------------------
predicted_pairs = set()

# التعامل بمرونة مع المفتاحين (conflicts_identified أو conflicts)
conflicts_list = answer.get('conflicts_identified') or answer.get('conflicts') or []

for pair in conflicts_list:
    if isinstance(pair, dict):
        pair = [pair.get('a') or pair.get('vessel_id_A'), pair.get('b') or pair.get('vessel_id_B')]
    elif isinstance(pair, str):
        pass

    if isinstance(pair, (list, tuple)) and len(pair) == 2 and all(isinstance(v, str) for v in pair) and pair[0] != pair[1]:
        predicted_pairs.add(tuple(sorted(pair)))
    else:
        violations.append(f'Malformed conflict pair: {pair}')

tp, fp, fn = (len(predicted_pairs & original_pairs), len(predicted_pairs - original_pairs),
              len(original_pairs - predicted_pairs))
precision = tp / (tp + fp) if tp + fp else float(not original_pairs)
recall = tp / (tp + fn) if tp + fn else 1.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

# ---- build the rescheduled timetable ---------
def same_value(col, given, original):
    if col == 'modules':
        given = str(given).strip('[] ').replace(' ', '')
        return valid_modules(given) and modules_as_set(given) == modules_as_set(original)
    return str(given).strip() == original


final = df.copy()
row_of = {v: i for i, v in enumerate(df.vessel_id)}
seen = set()

# التعامل بمرونة مع المفتاحين (rescheduling_changes أو changes)
changes_list = answer.get('rescheduling_changes') or answer.get('changes') or []

for ch in changes_list:
    vid = ch.get('vessel_id') if isinstance(ch, dict) else None
    if vid not in row_of:
        violations.append(f'Unknown vessel / malformed change: {ch}'); continue
    if vid in seen:
        violations.append(f'Duplicate change: {vid}'); continue
    seen.add(vid)
    i = row_of[vid]
    for col in ['berth', 'modules']:
        if col in ch and not same_value(col, ch[col], df.at[i, col]):
            echo_changed.append(vid)
            violations.append(f'{col} changed: {vid}')
    try:
        new_eta, new_etd = pd.Timestamp(ch['new_eta']), pd.Timestamp(ch['new_etd'])
        if pd.isna(new_eta) or pd.isna(new_etd) or new_eta.tzinfo or new_etd.tzinfo:
            raise ValueError('empty or timezone-aware date')
    except Exception as exc:
        violations.append(f'Unparseable date for {vid}: {exc}'); continue
    final.at[i, 'eta'], final.at[i, 'etd'] = new_eta, new_etd

if 'failed' in locals() and failed:
    violations.append(f'{len(failed)} batch(es) returned no valid answer')

# ---- schedule checks ------------------------------------------------------
final_pairs = find_conflicts(final)
remaining = original_pairs & final_pairs
new_pairs = final_pairs - original_pairs
delay = (final.eta - df.eta).dt.total_seconds() / 60
moved = delay != 0

bad_order    = final.vessel_id[final.etd <= final.eta].tolist()
bad_berth    = final.vessel_id[(final.berth != df.berth) | (final.modules != df.modules)].tolist()
bad_duration = final.vessel_id[(final.etd - final.eta) != (df.etd - df.eta)].tolist()
bad_early    = final.vessel_id[final.eta < df.eta].tolist()
for name, ids in [('ETD not after ETA', bad_order), ('Berth/modules changed', bad_berth),
                  ('Duration changed', bad_duration), ('ETA earlier than original', bad_early)]:
    violations += [f'{name}: {v}' for v in ids]


# ---- criterion 3: delay quality -------------------------------------------
def is_tight(i):
    """A moved vessel is tight if it starts exactly when a vessel on a shared module leaves."""
    r = final.loc[i]
    same = final[(final.berth == r.berth) & (final.etd == r.eta) & (final.index != i)]
    return any(modules_as_set(r.modules) & modules_as_set(m) for m in same.modules)


def greedy_reference(table):
    """First-come-first-served: each vessel takes the earliest conflict-free start >= its ETA."""
    out = table.copy()
    for _, group in table.groupby('berth'):
        placed = []
        for i, r in group.sort_values(['eta', 'vessel_id']).iterrows():
            mods, dur, start = modules_as_set(r.modules), r.etd - r.eta, r.eta
            placed = [p for p in placed if p[1] > r.eta]
            clash = True
            while clash:
                clash = [b for a, b, m in placed if m & mods and a < start + dur and start < b]
                if clash:
                    start = max(clash)
            placed.append((start, start + dur, mods))
            out.at[i, 'eta'], out.at[i, 'etd'] = start, start + dur
    return out


loose = [final.at[i, 'vessel_id'] for i in final.index[moved] if not is_tight(i)]
reference_delay = ((greedy_reference(df).eta - df.eta).dt.total_seconds() / 60).sum()
total_delay = delay[moved].sum()

# ---- scorecard ------------------------------------------------------------
failed_batches_count = len(failed) if 'failed' in locals() else 0
timetable_ok = not failed_batches_count and not any(v.startswith(('Unknown', 'Duplicate', 'Unparseable', 'Malformed'))
                                      for v in violations)
checks = [
    (1, 'Detect conflicts correctly', f1 == 1, f'P={precision:.2f}  R={recall:.2f}  F1={f1:.2f}  ({tp}/{len(original_pairs)} found, {fp} false)'),
    (2, 'Resolve conflicts', not remaining, f'{len(original_pairs) - len(remaining)}/{len(original_pairs)} original conflicts resolved'),
    (3, 'Minimize added delay', not remaining and not loose,
        f'{len(loose)} non-tight moves | total {total_delay/60:.1f} h vs greedy FCFS {reference_delay/60:.1f} h'),
    (4, 'Valid rescheduled timetable', timetable_ok, f'{len(final)} rows | {failed_batches_count} failed batches | {len(seen)} changes applied'),
    (5, 'ETA before ETD', not bad_order, f'{len(bad_order)} violations'),
    (6, 'Berth & modules preserved', not (bad_berth or echo_changed), f'{len(bad_berth) + len(echo_changed)} violations'),
    (7, 'Service duration preserved', not bad_duration, f'{len(bad_duration)} violations'),
    (8, 'No new conflicts', not new_pairs, f'{len(new_pairs)} new conflict pairs'),
    (9, 'ETA not earlier than original', not bad_early, f'{len(bad_early)} violations'),
]
checks.append((10, 'Final timetable valid & conflict-free',
               all(ok for n, _, ok, _ in checks if n in (2, 4, 5, 6, 7, 8, 9)),
               f'{len(violations)} constraint violations | {len(final_pairs)} conflicts in final timetable'))

scorecard = pd.DataFrame(checks, columns=['#', 'Criterion', 'Pass', 'Details'])
scorecard.insert(2, 'Status', scorecard.pop('Pass').map({True: '✅ PASS', False: '❌ FAIL'}))
print(f"SCORE: {scorecard.Status.str.contains('PASS').sum()}/10 criteria passed")
display(scorecard.style.hide(axis='index')
        .map(lambda s: f"color: {'#1a7f37' if 'PASS' in s else '#cf222e'}; font-weight: bold", subset=['Status'])
        .set_properties(subset=['Criterion', 'Details'], **{'text-align': 'left'}))

SCORE: 6/10 criteria passed


#,Criterion,Status,Details
1,Detect conflicts correctly,❌ FAIL,"P=0.00 R=0.00 F1=0.00 (0/47 found, 0 false)"
2,Resolve conflicts,❌ FAIL,0/47 original conflicts resolved
3,Minimize added delay,❌ FAIL,0 non-tight moves | total 0.0 h vs greedy FCFS 1378.4 h
4,Valid rescheduled timetable,✅ PASS,8675 rows | 0 failed batches | 0 changes applied
5,ETA before ETD,✅ PASS,0 violations
6,Berth & modules preserved,✅ PASS,0 violations
7,Service duration preserved,✅ PASS,0 violations
8,No new conflicts,✅ PASS,0 new conflict pairs
9,ETA not earlier than original,✅ PASS,0 violations
10,Final timetable valid & conflict-free,❌ FAIL,0 constraint violations | 47 conflicts in final timetable


In [9]:
tokens_in, tokens_out = sum(l['tokens_in'] for l in logs), sum(l['tokens_out'] for l in logs)
metrics = pd.DataFrame([
    ('Initial conflicts', len(original_pairs), 'pairs'),
    ('Detected (TP / FP / FN)', f'{tp} / {fp} / {fn}', 'pairs'),
    ('Detection precision / recall / F1', f'{precision:.3f} / {recall:.3f} / {f1:.3f}', ''),
    ('Remaining original conflicts', len(remaining), 'pairs'),
    ('New conflicts', len(new_pairs), 'pairs'),
    ('Conflict resolution rate (CRR)',
     round(100 * (len(original_pairs) - len(final_pairs)) / len(original_pairs), 1) if original_pairs else 100.0, '%'),
    ('Moved vessels', int(moved.sum()), 'vessels'),
    ('Total added delay', round(total_delay / 60, 2), 'hours'),
    ('Average added delay (AAD, per moved vessel)', round(delay[moved].mean(), 1) if moved.any() else 0.0, 'minutes'),
    ('Max added delay', round(delay.max() / 60, 2), 'hours'),
    ('Greedy FCFS reference delay', round(reference_delay / 60, 2), 'hours'),
    ('Constraint violations', len(violations), ''),
    ('Batches / failed / LLM calls', f"{len(logs)} / {len(failed)} / {sum(l['attempts'] for l in logs)}", ''),
    ('Execution time', seconds, 'seconds'),
    ('Tokens (in / out)', f'{tokens_in:,} / {tokens_out:,}', 'tokens'),
    ('Estimated cost', round((tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1e6, 4), 'USD'),
], columns=['Metric', 'Value', 'Unit'])
display(metrics.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

print('\nRescheduled vessels')
changed = df[moved].assign(new_eta=final.eta[moved], new_etd=final.etd[moved],
                           delay_min=delay[moved].round(0), tight=[is_tight(i) for i in df.index[moved]])
display(changed.reset_index(drop=True))

print('\nDetection per berth')
berth_of = dict(zip(df.vessel_id, df.berth))
per_berth = pd.DataFrame([(berth_of.get(p[0], '?'), p in original_pairs, p in predicted_pairs)
                          for p in original_pairs | predicted_pairs], columns=['berth', 'real', 'reported'])
display(per_berth.groupby('berth').agg(real=('real', 'sum'), reported=('reported', 'sum'),
                                       found=('real', lambda r: (r & per_berth.loc[r.index, 'reported']).sum()))
        .assign(false_alarms=lambda t: t.reported - t.found).astype(int))

if final_pairs:
    print('\nConflicts still in the final timetable')
    display(pd.DataFrame(sorted(final_pairs), columns=['vessel_A', 'vessel_B'])
            .assign(type=lambda t: ['original' if tuple(p) in original_pairs else 'NEW' for p in t.values]))
if violations:
    print('\nConstraint violations')
    display(pd.DataFrame({'violation': violations}))

Metric,Value,Unit
Initial conflicts,47,pairs
Detected (TP / FP / FN),0 / 0 / 47,pairs
Detection precision / recall / F1,0.000 / 0.000 / 0.000,
Remaining original conflicts,47,pairs
New conflicts,0,pairs
Conflict resolution rate (CRR),0.000000,%
Moved vessels,0,vessels
Total added delay,0.000000,hours
"Average added delay (AAD, per moved vessel)",0.000000,minutes
Max added delay,0.000000,hours



Rescheduled vessels


,vessel_id,berth,modules,eta,etd,new_eta,new_etd,delay_min,tight



Detection per berth


,real,reported,found,false_alarms
berth,,,,
01C,2,0,0,0
01D,1,0,0,0
19A,2,0,0,0
20A,1,0,0,0
20D,1,0,0,0
22B,1,0,0,0
22C,3,0,0,0
24B,3,0,0,0
29A,3,0,0,0



Conflicts still in the final timetable


,vessel_A,vessel_B,type
0,71442-1,71516-1,original
1,72596-1,74163-1,original
2,72710-1,73224-1,original
3,72720-1,73195-1,original
4,73066-1,73074-1,original
5,73108-1,73195-1,original
6,73334-1,73335-1,original
7,73581-1,74392-1,original
8,73892-1,73938-1,original
9,73933-1,73975-1,original


Optional: set `SAVE_RESULTS = True` to export the timetable, metrics and raw agent replies.

In [10]:
SAVE_RESULTS = True
if SAVE_RESULTS:
    stamp = time.strftime('%Y%m%d_%H%M%S')
    final.to_csv(f'rescheduled_timetable_{stamp}.csv', index=False)
    pd.concat([scorecard, metrics]).to_csv(f'baseline_metrics_{stamp}.csv', index=False)
    with open(f'agent_replies_{stamp}.json', 'w', encoding='utf-8') as f:
        json.dump(logs, f, indent=2, ensure_ascii=False, default=str)
    print('Saved:', stamp)

Saved: 20261014_190026
